# 04 — sitemap (the gate)

`public/sitemap.xml` of the fixture site: well-formed XML, absolute
URLs under the site origin, a full reciprocal hreflang matrix with
x-default, exclusions honored, robots.txt pointing at the sitemap.


In [ ]:
import jaen_testkit as k
k.start_run('04-sitemap')
print(k.CONFIG['repo_root'])

In [ ]:
import os, re

PUBLIC = k.site_path('public')
SITEMAP = os.path.join(PUBLIC, 'sitemap.xml')
locales = set(k.CONFIG['site_locales'])
default = k.CONFIG['site_default_locale']

with k.section('well-formedness'):
    with k.check('sitemap.xml parses') as c:
        if not os.path.isfile(SITEMAP):
            c.skip('no sitemap.xml — run 03 first')
        tree = c.require(k.xml_file(SITEMAP))
        entries = k.sitemap_entries(tree.root)
        c.expect_true(len(entries) > 0, '%d url entries' % len(entries))


In [ ]:
tree = k.xml_file(SITEMAP)
entries = k.sitemap_entries(tree.root) if tree.ok else []
locs = {e['loc'] for e in entries}

with k.section('URLs'):
    with k.check('every loc is absolute under one origin') as c:
        if not entries:
            c.skip('no entries')
        origins = {re.match(r'(https?://[^/]+)', e['loc']).group(1)
                   for e in entries if re.match(r'https?://', e['loc'])}
        c.expect_equal(len(origins), 1, 'single origin: %s' % sorted(origins))
        bad = [e['loc'] for e in entries if not e['loc'].startswith('http')]
        c.expect_equal(bad, [], 'no relative locs')

    with k.check('no duplicate locs') as c:
        if not entries:
            c.skip('no entries')
        c.expect_equal(len(locs), len(entries))

    with k.check('lastmod values are ISO-8601 when present') as c:
        if not entries:
            c.skip('no entries')
        bad = [e['lastmod'] for e in entries
               if e['lastmod'] and not re.match(r'^\d{4}-\d{2}-\d{2}', e['lastmod'])]
        c.expect_equal(bad, [], 'parsable dates')


In [ ]:
with k.section('hreflang matrix'):
    with k.check('localized entries carry the full alternate set + x-default') as c:
        if not entries:
            c.skip('no entries')
        incomplete = []
        with_alternates = [e for e in entries if e['alternates']]
        c.note('%d/%d entries carry alternates' % (len(with_alternates), len(entries)))
        for e in with_alternates:
            langs = {a['hreflang'] for a in e['alternates']}
            wanted = locales | {'x-default'}
            missing = {w for w in wanted
                       if not any(l == w or l.split('-')[0] == w for l in langs)}
            if missing:
                incomplete.append('%s missing %s' % (e['loc'], sorted(missing)))
        c.expect_equal(incomplete, [], 'full matrix everywhere')
        if incomplete:
            c.detail('\n'.join(incomplete[:10]))

    with k.check('alternates are reciprocal (every href is a loc)') as c:
        if not entries:
            c.skip('no entries')
        dangling = sorted({a['href'] for e in entries for a in e['alternates']
                           if a['href'] not in locs})
        c.expect_equal(dangling, [], 'no dangling alternates')
        if dangling:
            c.detail('\n'.join(dangling[:10]))

    with k.check('x-default points at the default locale variant') as c:
        if not entries:
            c.skip('no entries')
        wrong = []
        for e in entries:
            xdef = [a for a in e['alternates'] if a['hreflang'] == 'x-default']
            defs = [a for a in e['alternates']
                    if a['hreflang'] == default or a['hreflang'].split('-')[0] == default]
            if xdef and defs and xdef[0]['href'] != defs[0]['href']:
                wrong.append(e['loc'])
        c.expect_equal(wrong, [], 'x-default == default-locale href')


In [ ]:
with k.section('exclusions + robots'):
    with k.check('system routes are excluded') as c:
        if not entries:
            c.skip('no entries')
        leaked = [e['loc'] for e in entries
                  if re.search(r'/(cms|login|logout|settings|signup|password_reset|emailwerk)(/|$)', e['loc'])]
        c.expect_equal(leaked, [], 'no system routes in the sitemap')

    with k.check('no 404/500/dev pages') as c:
        if not entries:
            c.skip('no entries')
        leaked = [e['loc'] for e in entries if re.search(r'/(404|500|dev-404)', e['loc'])]
        c.expect_equal(leaked, [], 'clean')

    with k.check('robots.txt references the sitemap') as c:
        robots = k.read_text(os.path.join(PUBLIC, 'robots.txt'))
        if robots is None:
            c.fail('robots.txt missing', abort=True)
        c.expect_contains(robots, 'Sitemap:')
        c.expect_contains(robots, '/sitemap.xml')


In [ ]:
k.summary()
k.save_results('results-04-sitemap.json')
rc = k.verdict()
assert rc == 0, 'run has FAILures — see the summary above'